# **OPTION D- AI Resume Screening & Shortlisting System**

**It Should have:**



1.  Resume Analyzer Agent

2.   Skill Evaluator Agent

3. HR Decision Writer Agent

**Step 1:** Install Packages

In [ ]:
!pip uninstall -y crewai crewai-tools litellm
!pip install crewai litellm apscheduler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.6 M

**Step 2:** Configure LLM

In [ ]:
import os
import logging
from crewai import Agent, Task, Crew, LLM

# --------------------------------------------------
# SET ONLY GROQ API KEY

# --------------------------------------------------
os.environ["GROQ_API_KEY"] = "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"


os.environ["LITELLM_LOG"] = "CRITICAL"
os.environ["LITELLM_DISABLE_PROXY"] = "true"

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
# --------------------------------------------------
# CREATE GROQ LLM OBJECT
# This avoids LiteLLM fallback error
# --------------------------------------------------
groq_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.7
)

print("Groq Connected Successfully ")


Groq Connected Successfully 


**Explanation:**

*   Set API Key: Provide GROQ_API_KEY for authentication.
*   Suppress Logging: Disable LiteLLM proxy and set log level to CRITICAL.

*  Create LLM Object: Initialize Groq LLM to prevent LiteLLM fallback errors.



**Step 3:** Create HR Agents

In [ ]:
# --------------------------------------------------
# SAMPLE RESUME INPUT
# --------------------------------------------------
resume_text = """
Name: Rahul Sharma
Experience: 3 years as Backend Developer
Skills: Python, Django, REST APIs, SQL, Docker
Worked on microservices architecture and deployed apps on AWS.
"""

job_description = """
Looking for Backend Developer with:
- Strong Python skills
- REST API experience
- Cloud deployment knowledge
- Microservices experience preferred
"""

# --------------------------------------------------
# AGENT 1 — Resume Analyzer
# --------------------------------------------------
analyzer_agent = Agent(
    role="Resume Analyzer",
    goal="Extract key skills, experience level, and strengths from resume.",
    backstory="Expert in HR resume screening.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)

# --------------------------------------------------
# AGENT 2 — Skill Evaluator
# --------------------------------------------------
evaluator_agent = Agent(
    role="Skill Evaluator",
    goal="Compare resume with job description and evaluate match percentage.",
    backstory="Technical hiring specialist.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)

# --------------------------------------------------
# AGENT 3 — HR Decision Writer
# --------------------------------------------------
decision_agent = Agent(
    role="HR Decision Writer",
    goal="Generate final hiring recommendation report.",
    backstory="Senior HR manager making hiring decisions.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)


**Explanation:**

*   Resume Analyzer: Extract key skills, experience, and strengths from the resume.

*   Skill Evaluator: Compare resume with job description and calculate match percentage.
*  HR Decision Writer: Generate final hiring recommendation report.


**Step 4:** Create HR Tasks

In [ ]:
# --------------------------------------------------
# TASK 1 — Analyze Resume
# --------------------------------------------------
analysis_task = Task(
    description=f"""
Analyze the following resume:

{resume_text}

Extract:
- Key Skills
- Years of Experience
- Core Strengths
""",
    expected_output="Structured resume analysis.",
    agent=analyzer_agent
)

# --------------------------------------------------
# TASK 2 — Evaluate Skill Match
# --------------------------------------------------
evaluation_task = Task(
    description=f"""
Compare the analyzed resume with this job description:

{job_description}

Provide:
- Skill match percentage
- Missing skills (if any)
- Overall technical fit (Good/Average/Weak)
""",
    expected_output="Skill match evaluation report.",
    agent=evaluator_agent
)

# --------------------------------------------------
# TASK 3 — Final HR Decision
# --------------------------------------------------
decision_task = Task(
    description="""
Based on previous analysis and evaluation,
Generate final hiring recommendation including:
- Shortlisting decision (Yes/No)
- Reason
- Suggested next step
""",
    expected_output="Final HR hiring recommendation.",
    agent=decision_agent
)


**Explanation:**

* Analyze Resume: Extract skills, experience, and strengths from the candidate’s resume.

*  Evaluate Skill Match: Compare resume with job description and assess fit.
*  Final HR Decision: Generate a hiring recommendation with reasoning and next steps.



**Step 5:** Launch HR Crew

In [ ]:
# --------------------------------------------------
# CREATE CREW
# --------------------------------------------------
crew = Crew(
    agents=[analyzer_agent, evaluator_agent, decision_agent],
    tasks=[analysis_task, evaluation_task, decision_task],
    verbose=True
)

# --------------------------------------------------
# RUN CREW
# --------------------------------------------------
result = crew.kickoff()

print("\n========== FINAL HIRING REPORT ==========\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d41c5f50-12d1-40cc-88a3-edea5806f8d6                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the following resume:                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Name: Rahul Sharma                                                                                             │
│  Experience: 3 years as Backend Developer                                                                       │
│  Skills: Python, Django, REST APIs, SQL, Docker                                                                 │
│  Worked on microservices architecture and deployed apps on AWS.                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Extract:                                                                                                       │
│  - Key Skills                                                                                                   │
│  - Years of Experience                                                                                          │
│  - Core Strengths                                                                                               │
│                                                                                                                 │
│  ID: a4130770-d201-4b8a-b932-3bf1c4950882                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyzer                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the following resume:                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Name: Rahul Sharma                                                                                             │
│  Experience: 3 years as Backend Developer                                                                       │
│  Skills: Python, Django, REST APIs, SQL, Docker                                                                 │
│  Worked on microservices architecture and deployed apps on AWS.                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Extract:                                                                                                       │
│  - Key Skills                                                                                                   │
│  - Years of Experience                                                                                          │
│  - Core Strengths                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyzer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Structured Resume Analysis**                                                                                 │
│                                                                                                                 │
│  **Name:** Rahul Sharma                                                                                         │
│                                                                                                                 │
│  **Key Skills:**                                                                                                │
│                                                                                                                 │
│  1. **Programming Languages:** Python                                                                           │
│  2. **Frameworks and Libraries:** Django                                                                        │
│  3. **APIs:** REST APIs                                                                                         │
│  4. **Database Management:** SQL                                                                                │
│  5. **Containerization:** Docker                                                                                │
│  6. **Cloud Platforms:** AWS                                                                                    │
│                                                                                                                 │
│  **Years of Experience:** 3 years                                                                               │
│                                                                                                                 │
│  **Core Strengths:**                                                                                            │
│                                                                                                                 │
│  1. **Backend Development:** Experienced in backend development with 3 years of experience.                     │
│  2. **Microservices Architecture:** Proficient in designing and implementing microservices architecture.        │
│  3. **Cloud Deployment:** Skilled in deploying applications on AWS.                                             │
│  4. **Technical Expertise:** Strong understanding of Python, Django, and SQL.                                   │
│  5. **Containerization:** Knowledge of Docker for containerization.                                             │
│  6. **API Development:** Experienced in developing REST APIs.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│  Analyze the following resume:                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Name: Rahul Sharma                                                                                             │
│  Experience: 3 years as Backend Developer                                                                       │
│  Skills: Python, Django, REST APIs, SQL, Docker                                                                 │
│  Worked on microservices architecture and deployed apps on AWS.                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Extract:                                                                                                       │
│  - Key Skills                                                                                                   │
│  - Years of Experience                                                                                          │
│  - Core Strengths                                                                                               │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Resume Analyzer                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HR Decision Writer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Final HR Hiring Recommendation**                                                                             │
│                                                                                                                 │
│  **Shortlisting Decision:** Yes                                                                                 │
│                                                                                                                 │
│  **Reason:** Rahul Sharma's resume shows a strong match with the job description, with a skill match            │
│  percentage of 90%. He has relevant experience in backend development, microservices architecture, cloud        │
│  deployment, and API development, which are all critical requirements for the position. His technical           │
│  expertise in Python, Django, and SQL, along with his experience with Docker and AWS, make him a strong         │
│  candidate for the job.                                                                                         │
│                                                                                                                 │
│  **Suggested Next Step:**                                                                                       │
│                                                                                                                 │
│  Based on the analysis, it is recommended to invite Rahul Sharma for an interview to further assess his skills  │
│  and experience. The interview should focus on his experience with specific frameworks and libraries, as well   │
│  as his understanding of the company's technology stack and requirements. Additionally, it would be beneficial  │
│  to have him complete a technical assessment or coding challenge to evaluate his problem-solving skills and     │
│  ability to work with the company's technology.                                                                 │
│                                                                                                                 │
│  **Additional Recommendations:**                                                                                │
│                                                                                                                 │
│  * Ensure that the interview panel includes technical experts who can ask in-depth questions about the          │
│  candidate's experience and skills.                                                                             │
│  * Consider providing the candidate with a technical assessment or coding challenge as part of the interview    │
│  process to evaluate his skills and problem-solving ability.                                                    │
│  * Make sure to communicate clearly with the candidate about the company's expectations and requirements for    │
│  the position.                                                                                                  │
│                                                                                                                 │
│  **Rating:** (Highly Recommended)                                                                               │
│                                                                                                                 │
│  This recommendation is based on the strong match betwe

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│  Based on previous analysis and evaluation,                                                                     │
│  Generate final hiring recommendation including:                                                                │
│  - Shortlisting decision (Yes/No)                                                                               │
│  - Reason                                                                                                       │
│  - Suggested next step                                                                                          │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  HR Decision Writer                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d41c5f50-12d1-40cc-88a3-edea5806f8d6                                                                           │
│  Final Output: **Final HR Hiring Recommendation**                                                               │
│                                                                                                                 │
│  **Shortlisting Decision:** Yes                                                                                 │
│                                                                                                                 │
│  **Reason:** Rahul Sharma's resume shows a strong match with the job description, with a skill match            │
│  percentage of 90%. He has relevant experience in backend development, microservices architecture, cloud        │
│  deployment, and API development, which are all critical requirements for the position. His technical           │
│  expertise in Python, Django, and SQL, along with his experience with Docker and AWS, make him a strong         │
│  candidate for the job.                                                                                         │
│                                                                                                                 │
│  **Suggested Next Step:**                                                                                       │
│                                                                                                                 │
│  Based on the analysis, it is recommended to invite Rahul Sharma for an interview to further assess his skills  │
│  and experience. The interview should focus on his experience with specific frameworks and libraries, as well   │
│  as his understanding of the company's technology stack and requirements. Additionally, it would be beneficial  │
│  to have him complete a technical assessment or coding challenge to evaluate his problem-solving skills and     │
│  ability to work with the company's technology.                                                                 │
│                                                                                                                 │
│  **Additional Recommendations:**                                                                                │
│                                                                                                                 │
│  * Ensure that the interview panel includes technical experts who can ask in-depth questions about the          │
│  candidate's experience and skills.                                                                             │
│  * Consider providing the candidate with a technical assessment or coding challenge as part of the interview    │
│  process to evaluate his skills and problem-solving ability.                                                    │
│  * Make sure to communicate clearly with the candidate about the company's expectations and requirements for    │
│  the position.                                                                                                  │
│                                                                                                                 │
│  **Rating:** (Highly Recommended)                     


========== FINAL HIRING REPORT ==========

**Final HR Hiring Recommendation**

**Shortlisting Decision:** Yes

**Reason:** Rahul Sharma's resume shows a strong match with the job description, with a skill match percentage of 90%. He has relevant experience in backend development, microservices architecture, cloud deployment, and API development, which are all critical requirements for the position. His technical expertise in Python, Django, and SQL, along with his experience with Docker and AWS, make him a strong candidate for the job.

**Suggested Next Step:**

Based on the analysis, it is recommended to invite Rahul Sharma for an interview to further assess his skills and experience. The interview should focus on his experience with specific frameworks and libraries, as well as his understanding of the company's technology stack and requirements. Additionally, it would be beneficial to have him complete a technical assessment or coding challenge to evaluate his problem-solving skill

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Explanation:**

*  Assemble Crew: Combine resume analyzer, skill evaluator, and decision writer agents.


* Execute Tasks: Run tasks sequentially—analyze resume → evaluate skills → generate hiring recommendation.
*   Output: Kickoff workflow and display the final HR hiring report.

